# DATA 612 — Research Discussion 2: Recommendations at Scale

**Name:** Noah Collin  **Course:** DATA 612

This video introduced me abruptly to some important concepts, many of which I had to google further to fully understand.

I first had to search for the difference between explicit and implicit ratings. It sounds intuitive at first, like for videos, you can understand that when watching a YouTube video, even if you don't "like" it (with the explicit like button), you're still interacting with the site and generating ad revenue. Once I dug in, the thing that surprised me is that the choice of what signal you optimize basically decides what behavior you breed: optimize for clicks and you get clickbait; optimize for raw watch time and you get autoplay rabbit holes. The weighting decision is where "user satisfaction" quietly gets swapped out for "whatever number you happened to put a big coefficient on." I'd assumed a "like" is just a stronger signal than a watch, but the Covington et al. (2016) YouTube deep-learning recommender paper deliberately went the other way — they trained on watch time rather than explicit clicks/likes, precisely because explicit clicks are sparse, biased, and gameable, while watch time is dense and (they argued) closer to real satisfaction. So a like isn't automatically "stronger" — it's sparser and more explicit, which is not the same thing. Explicit signals have low coverage and selection bias (only certain users click the button, on certain content); implicit signals have high coverage but noisier intent. They measure different things, and "stronger" depends entirely on what you're trying to predict.

Summarily, if you mentally model the desired behavior wrong, then your vectors learn an ultimately undesirable signal. In other words, modeling implicit feedback is its own modeling problem, and then modeling a recommender system on that is its own downstream modeling problem. If you model implicit user engagement in an incorrect way, your recommender system is learning something well that doesn't really exist. I for one would have wagered, based purely of intuition, that a "like" (button press) is a stronger signal that merely watching. But research as cited above disproved that.

I have never seriously worked with distributed computing. But as I searched about Hadoop and spark, it reminded me of a concept I did know a bit about that I'll mention at the end. I learned that Hadoop has a methodology that does distributed work, but it reads and writes to disk much more often. In fact, for a map reduce, which it was basically meant for, it would frequently divide these two portions of an algorithm, where the map happens on machine A, which has to write its results to disk, and then machine B reads that and does the reduce. Spark decided to keep data in memory only on machine A for more operations. You could implement checkpoints with disk writes, but at a cost of slow downs. The question obviously becomes if you skip checkpoints and a machine crashes, what happens? Because everything's in memory and nothing is on disk, a different machine would just take start over at the first step via "lineage" or the last check point. Checkpoints are typically written to 3+ machines for fault tolerance. So one criticism of spark is that it's actually only faster if the data from the chunked data actually fits on ram of all the downstream machines. Let's say you distribute work over 1000 computers, and each machine's chunk is going to be 60 gb of data. If all machines have 64+ gb of ram, then the whole operation will fly at ram speed. But if even one machine has 32 gb of ram, that machine has to do data onloading and offloading on to disk internally. Since it can't process the 60gb of data all in ram at once, it has to internally pass things on and off its own disk, and the whole system waits for that to happen. Another thing to know is that there's a "driver" machine, that just handles the flight control and lineage operations. But that's a point of failure that's not handled like the distributed computing is. So you could get a driver driver that can restart the driver if it fails and it can resume from a checkpoint. But if you are doing them infrequently, a driver crash (not to be confused with a hardware driver) can be a single point of failure for the whole distributed computing.

This actually reminded me of something I'd read months ago about how distributed AI systems handle the very same orchestrator problem. The "hands" are the worker instances that do the actual work and can fail, and the "brain" (the orchestrator/driver) can restart them — but if the brain itself fails, how does it resume? The trick is a harness: the brain periodically writes its state out to a file, so the harness can restart the braindead (pun intended) orchestrator from that state, and the brain in turn has the ability to restart the harness. Think of the spider man meme where each part is pointing at the other part, but they can all kind of reboot each other, with a higher-level supervisor keeping an eye on the whole thing and able to restart it at that level if needed. It's the exact same driver/checkpoint idea as Spark, just pointed at distributed AI work — the whole point being you get fault tolerance without one fragile component taking everything down with it (intend your puns!).

It does sound like there will be continuing evolution of this. I'm not in the weeds of distributed computing, but I think everyone will have to keep abreast of the way the wind is blowing here. I also feel for people who need to manage local distributed computing instead of relying on a service to oversee it. It sounds like even though fault tolerance is a design principle, there's still a lot of gotchas to think about.